##Assignment 5- Cloud Data Engineering:

###Theme: Enterprise-Grade Cloud Data Platform Implementation

###PART A – AZURE DATA ENGINEERING ASSIGNMENT


#### Logging

In [0]:
import datetime
def log(message, status="INFO"):
    print(f"[{status}] {datetime.datetime.now()} - {message}")
log("Notebook execution started")
start_time = datetime.datetime.now()

##Task 1: Cloud Foundation Setup (Infra + Security) 


####Part 2: Option3: Mount ADLS securely in Databricks

In [0]:
storage_account = "assgin5dlsgen2aman"
container = "raw"
bronze_container = "bronze"
silver_container="silver"
gold_container="gold"

configs = {
  "fs.azure.account.auth.type": "OAuth",
  "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
  "fs.azure.account.oauth2.client.id": "994a81fb-7814-4358-aff3-e1cce42cbc5c",
  "fs.azure.account.oauth2.client.secret": "<REDACTED>",
  "fs.azure.account.oauth2.client.endpoint": "https://login.microsoftonline.com/14f7a50f-038d-49e7-ad33-d6c457655b8a/oauth2/token"
}

for key, value in configs.items():
    spark.conf.set(f"{key}.{storage_account}.dfs.core.windows.net", value)

# Use this path instead of /mnt/raw in downstream cells
adls_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/"
bronze_path = f"abfss://{bronze_container}@{storage_account}.dfs.core.windows.net/"
silver_path = f"abfss://{silver_container}@{storage_account}.dfs.core.windows.net/"
gold_path = f"abfss://{gold_container}@{storage_account}.dfs.core.windows.net/"
print(f"ADLS path configured: {adls_path}")
dbutils.fs.ls(adls_path)

Logging

In [0]:
log("storga configuration completed")

Checking that mounting is successful or not

In [0]:
dbutils.fs.ls(adls_path)
dbutils.fs.ls(bronze_path)
dbutils.fs.ls(silver_path)
dbutils.fs.ls(gold_path)

####TASK 2 – Ingestion Framework (ADF + Databricks) 

####Part 2: Parameterize: File path, Date, Environment (Dev/Test/Prod)

In [0]:
dbutils.widgets.text("file_path", "")
dbutils.widgets.text("date", "")
dbutils.widgets.text("env", "")
 
file_path = dbutils.widgets.get("file_path")
date = dbutils.widgets.get("date")
env = dbutils.widgets.get("env")
 
print(file_path, date, env)
 

logging

In [0]:
log(f"parameters passed: file_path={file_path}, date={date}, env={env}")

####TASK 1: Part 3: Validate: Reading data from ADLS via PySpark
####TASK 2: Part 3: Handle: Schema Drift

In [0]:
log("reading data from ADLS")

In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("mode", "PERMISSIVE") \
    .load(f"{adls_path}sales/{file_path}")

df.show()

In [0]:
log("Data loaded successfully")

####TASK 5 – Monitoring & Governance 
####Part 2:  Data Quality Checks

In [0]:
log("Running data quality checks")
# 1. Null check
if df.filter("order_id IS NULL").count() > 0:
    log("Null values found in order_id", "ERROR")
# 2. Duplicate check
if df.count() != df.dropDuplicates().count():
    log("Duplicate records found")
# 3. Business rule
if df.filter("price <= 0").count() > 0:
    log("Invalid price detected", "ERROR")
log("Data quality checks completed")
 

###Task 3: Lakehouse Medallion Architecture
Implement:  
Raw → Bronze → Silver → Gold (Delta Lake) 

Bronze:  
• Raw ingestion  
• Add ingestion timestamp  
• Store as Delta  

In [0]:
from pyspark.sql.functions import current_timestamp 
# Add ingestion timestamp
df = df.withColumn("ingestion_time", current_timestamp())
 
# Write to Bronze (Delta)
df.write.format("delta") \
    .mode("append") \
    .save(f"{bronze_path}sales/")

In [0]:
log("Data transformation started")

Silver:  
• Clean nulls  
• Standardize schema  
• Remove duplicates  
• Apply business rules  

In [0]:
df = spark.read.format("delta") \
    .load(f"{bronze_path}sales/")
 
# Remove nulls
df = df.dropna()
# Remove duplicates
df = df.dropDuplicates()
# Standardize
df = df.withColumnRenamed("product_name", "product")
# Write to Silver
df.write.format("delta") \
    .mode("overwrite") \
    .save(f"{silver_path}sales/")

In [0]:
log("Data transformation completed")

Gold:                         
• Business KPIs               
• Aggregations  
• Partitioned tables  
 
 Include: Delta merge, Upsert logic, Time travel demo  

In [0]:
log("writing data to Delta/ADLS")

In [0]:
from pyspark.sql.functions import sum
 
df = spark.read.format("delta") \
    .load(f"{silver_path}sales/")
 
# KPI: total sales per region
gold_df = df.groupBy("region") \
    .agg(sum("price").alias("total_sales"))
 
# Write to Gold
gold_df.write.format("delta") \
    .mode("overwrite") \
    .save(f"{gold_path}sales/")

In [0]:
log("Data writen successfully")

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

# Step 1: Deduplication
window = Window.partitionBy("order_id").orderBy(col("order_date").desc())

df_dedup = df.withColumn("row_num", row_number().over(window)) \
             .filter("row_num = 1") \
             .drop("row_num")

# Step 2: Load Delta table
delta_table = DeltaTable.forPath(
    spark,
    f"{silver_path}sales/"
)

# Step 3: Merge
delta_table.alias("target").merge(
    df_dedup.alias("source"),
    "target.order_id = source.order_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

In [0]:
df_old = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load(f"{silver_path}sales/")
 
df_old.show()

###Task-4

####Step1: Before Optimisation


In [0]:
df = spark.read.format("delta") \
    .load(f"{gold_path}sales/")

df.show()

In [0]:
import time
df = spark.read.format("delta") \
    .load(f"{silver_path}sales/")
start = time.time()

df.groupBy("region").sum("price").show()

end = time.time()

before_optimization_runtime = end - start

print("Time Taken:", before_optimization_runtime)

#### Step 2: Apply Partitioning

In [0]:
# Step 1: Create Gold data (aggregation)
df_gold = df.groupBy("region").sum("price") \
            .withColumnRenamed("sum(price)", "total_sales")
# Step 2: Write to Gold layer
df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("region") \
    .save(f"{gold_path}sales/")

####OPTIMIZE (File compaction)

In [0]:
spark.sql("""
OPTIMIZE delta.`abfss://gold@assgin5dlsgen2aman.dfs.core.windows.net/sales/`
""")

#### Step 4: Z-Ordering

In [0]:
spark.sql("""
OPTIMIZE delta.`abfss://gold@assgin5dlsgen2aman.dfs.core.windows.net/sales/`
ZORDER BY (total_sales)
""")

####Step 5: Cache usage

In [0]:
df = spark.read.format("delta") \
    .load(f"{gold_path}sales/")
df.cache()
# cache trigger
df.count()  

####Step 6: Explain Plan

In [0]:
df = spark.read.format("delta") \
    .load(f"{silver_path}sales/")

df.groupBy("region").sum("price").explain(True)

####Step 7: After Optimization runtime(Measure again)

In [0]:
start = time.time()
df.groupBy("region").sum("price").show()
end = time.time()
after_optimization_runtime = end - start
print("Optimized Time:", after_optimization_runtime)

##Performance Comparision Report

In [0]:
print("1. Before optimization, Time taken = ", before_optimization_runtime)
print("2. After optimization, Time taken = ", after_optimization_runtime)
print("3. Improvement = ", before_optimization_runtime - after_optimization_runtime)


## Task 5- Monitoring & Governance

### Part 3: Document:

### Failure Recovery Process
In this pipeline, failure handling is designed to ensure smooth execution and quick recovery without impacting downstream processes.
- 1. File Availability Check (Pre-validation)
Before processing the data, the pipeline checks whether the input file exists in the landing layer using the Get Metadata activity in Azure Data Factory.
  - If the file exists → pipeline continues execution
  If the file is missing → pipeline
  - skips processing using the False branch of If Condition
This prevents unnecessary failures due to missing input files.
- 2. Activity-Level Monitoring
Each activity in the pipeline (Copy Activity and Databricks Notebook) is monitored using the ADF Monitor tab.
  - Success and failure status can be tracked
  - Error messages are captured for debugging
  - Execution time is recorded
- 3. Notebook Error Handling
The Databricks notebook uses try-catch (exception handling) to capture runtime errors.
  - If an error occurs → it is logged with error details
  - Logs help in identifying issues like incorrect paths, schema changes, or data problems
- 4. Logging Framework
A basic logging framework is implemented in the notebook.
  - Logs are generated at each major step:
      - Pipeline start
      - Data read
      - Transformation
      - Data write
      - Pipeline end
  - Logs include timestamps for better traceability
- 5. Recovery Steps
In case of failure, the following steps are followed:
    - 1. Check pipeline status in ADF Monitor
    - 2. Identify failed activity
    - 3. Review error logs in Databricks
    - 4. Fix the issue (data/config/code)
    - 5. Re-run the pipeline using Trigger

### SLA (Service Level Agreement)
**Handling**
The pipeline follows a defined SLA to ensure timely data processing and availability.
- 1. Schedule
  - The pipeline is triggered daily using a schedule trigger in Azure Data Factory
  - Example: Runs every day at a fixed time (e.g., 8:00 AM)
- 2. Expected Completion Time
  - The pipeline is expected to complete within a defined time window (e.g., 15–30 minutes)
  - This includes data ingestion, processing, and loading
- 3. Monitoring Against SLA
  - Execution time is tracked using:
    - ADF Monitor tab
    - Databricks job run details
  - Any delay beyond expected time is considered an SLA breach
- 4. Handling SLA Breach
If the pipeline does not complete within the expected time:
  - Check pipeline status in ADF
  - Identify bottlenecks (slow read/write, large data, etc.)
  - Optimize performance (partitioning, caching, etc.)
  - Re-run the pipeline if required
- 5. Scalability Consideration
The pipeline is designed to scale with increasing data volume using:
  - Distributed processing (Databricks)
  - Delta Lake optimizations
  - Partitioning strategies

In [0]:
log(f"Notebook execution completed")